In [ ]:
!wget https://zenodo.org/record/4068196/files/mridangam_stroke_1.5.zip?download=1 -O mridangam_dataset.zip


!unzip -q mridangam_dataset.zip -d mridangam_stroke_dataset/


!rm mridangam_dataset.zip

print("Dataset ready!")

--2026-04-14 00:29:00--  https://zenodo.org/record/4068196/files/mridangam_stroke_1.5.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 137.138.52.235, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/4068196/files/mridangam_stroke_1.5.zip [following]
--2026-04-14 00:29:01--  https://zenodo.org/records/4068196/files/mridangam_stroke_1.5.zip
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 130280712 (124M) [application/octet-stream]
Saving to: ‘mridangam_dataset.zip’

mridangam_dataset.z 100%[===================>] 124.25M  24.0MB/s    in 6.3s    

2026-04-14 00:29:08 (19.7 MB/s) - ‘mridangam_dataset.zip’ saved [130280712/130280712]



In [2]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models



In [3]:
BASE_PATH = "mridangam_stroke_dataset/mridangam_stroke_1.5/"
tonics = [d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))]
print(f"Tonic folders found: {tonics}")

Tonic folders found: ['C#', 'D#', 'C', 'E', 'B', 'D']


In [4]:
def extract_spectrogram_fixed(file_path, target_width=64):
    # Load and normalize
    audio, sr = librosa.load(file_path, sr=22050)
    audio = librosa.util.normalize(audio)
    audio, _ = librosa.effects.trim(audio)

    # Generate Mel Spectrogram
    spectrogram = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=128, hop_length=256, n_fft=1024
    )
    spectrogram_db = librosa.power_to_db(spectrogram, ref=np.max)

    # Consistency Check: Resize to exactly target_width
    if spectrogram_db.shape[1] < target_width:
        pad_width = target_width - spectrogram_db.shape[1]
        spectrogram_db = np.pad(spectrogram_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
    else:
        spectrogram_db = spectrogram_db[:, :target_width]

    return spectrogram_db

In [8]:
X_spec = []
y_stroke = []

print("Extracting Mel-Spectrograms using STROKE labels...")

for tonic in tonics:
    folder_path = os.path.join(BASE_PATH, tonic)
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.wav'):
            # NEW SPLIT LOGIC for: 224051__akshaylaya__cha-b-017.wav
            # 1. Split by double underscores
            parts = file_name.split('__')

            if len(parts) >= 3:
                # parts[2] is 'cha-b-017.wav'
                # 2. Split that part by the first dash to get just the stroke
                stroke_segment = parts[2].split('-')[0]

                feat = extract_spectrogram_fixed(os.path.join(folder_path, file_name))
                X_spec.append(feat)
                y_stroke.append(stroke_segment)

# Final formatting
X_spec = np.array(X_spec).reshape(-1, 128, 64, 1)
le_stroke = LabelEncoder()
y_encoded = le_stroke.fit_transform(y_stroke)
num_classes = len(le_stroke.classes_)

X_train, X_test, y_train, y_test = train_test_split(X_spec, y_encoded, test_size=0.2, random_state=42)

print(f"Success! Correct Stroke categories found: {le_stroke.classes_}")

Extracting Mel-Spectrograms using STROKE labels...


/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=507
  warnings.warn(


Success! Correct Stroke categories found: ['akshaylaya' 'bheem' 'cha' 'dheem' 'dhin' 'num' 'ta' 'tha' 'tham' 'thi'
 'thom']


In [10]:
def build_mridangam_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        # Dense
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

In [11]:
model_stroke = build_mridangam_cnn((128, 64, 1), num_classes)
model_stroke.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model_stroke.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=[reduce_lr, early_stop]
)

Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 153s 841ms/step - accuracy: 0.5524 - loss: 1.6381 - val_accuracy: 0.6626 - val_loss: 0.9692 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 200s 831ms/step - accuracy: 0.7631 - loss: 0.6426 - val_accuracy: 0.7572 - val_loss: 0.7108 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 150s 857ms/step - accuracy: 0.8169 - loss: 0.5049 - val_accuracy: 0.9348 - val_loss: 0.2493 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 145s 827ms/step - accuracy: 0.8712 - loss: 0.3727 - val_accuracy: 0.9398 - val_loss: 0.2098 - learning_rate: 5.0000e-04
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 149s 850ms/step - accuracy: 0.8914 - loss: 0.3266 - val_accuracy: 0.9305 - val_loss: 0.1953 - learning_rate: 5.0000e-04
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 199s 831ms/step - accuracy: 0.8907 - loss: 0.3228 - val_accuracy: 0.9484 - val_loss: 0.2124 - learning_rate: 5.0000e-04
Epoch 7/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 145s 829ms/s

In [13]:
!apt-get install -y ffmpeg libavcodec-extra-qq
from google.colab import files

print("Upload your recorded Mridangam stroke (mp4, m4a, wav):")
uploaded = files.upload()

for filename in uploaded.keys():
    try:
        # Predict using same fixed extraction function
        user_feat = extract_spectrogram_fixed(filename)
        user_feat = user_feat.reshape(1, 128, 64, 1)

        preds = model_stroke.predict(user_feat, verbose=0)
        label = le_stroke.inverse_transform([np.argmax(preds)])[0]
        confidence = np.max(preds) * 100

        print(f"\n--- Analysis for: {filename} ---")
        print(f"Predicted Stroke: {label}")
        print(f"Confidence: {confidence:.2f}%")

    except Exception as e:
        print(f"Error analyzing file: {e}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package libavcodec-extra-qq
Upload your recorded Mridangam stroke (mp4, m4a, wav):


Saving try classification.mp4 to try classification (2).mp4


/tmp/ipykernel_2736/4120605615.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=22050)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



--- Analysis for: try classification (2).mp4 ---
Predicted Stroke: thom
Confidence: 66.05%
